# Exploração textual

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/02_exploracao_textual.ipynb)
## Tokenização e normalização
Tokenizar segmenta por regra. Minúsculas, pontuação e stopwords podem apagar distinções; preserve o texto e documente decisões. Neste exemplo, os tokens são calculados separadamente por documento para não criar contextos ou n-gramas entre o fim de um texto e o início do seguinte.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import math
import re
from collections import Counter

import pandas as pd

dados = pd.read_csv("dados/documentos.csv")

def tokenizar(texto):
    return re.findall(r"[a-záàâãéêíóôõúç]+", texto.lower())

dados["tokens"] = dados["texto"].map(tokenizar)
todos_tokens = [token for documento in dados["tokens"] for token in documento]
dados[["id_documento", "tokens"]].head(2)

## Frequências absoluta e relativa

Se $t_i$ é o token na posição $i$, a frequência de uma palavra $w$ é:

$$
f(w)=\sum_{i=1}^{N}\mathbf{1}(t_i=w),
\qquad
p(w)=\frac{f(w)}{N}.
$$

O valor de $p(w)$ só é interpretável quando $N$ está declarado. Neste
experimento mostraremos dois denominadores:

| Medida | Denominador |
|---|---|
| frequência relativa entre todos os tokens | $N$, antes de retirar stopwords |
| frequência relativa entre tokens de conteúdo | $N_c$, depois de retirar stopwords |

Remover stopwords apenas da tabela, mas manter $N$ como denominador, responde a
uma pergunta diferente de recalcular a proporção dentro do vocabulário filtrado.

In [ ]:
stopwords = {"a", "o", "e", "de", "do", "da", "como", "em", "nas", "um", "uma", "também"}
frequencias = Counter(todos_tokens)
tokens_conteudo = [token for token in todos_tokens if token not in stopwords]
frequencias_conteudo = Counter(tokens_conteudo)
total_tokens = len(todos_tokens)
total_tokens_conteudo = len(tokens_conteudo)

tabela_frequencias = pd.DataFrame([
    {
        "palavra": palavra,
        "frequencia": frequencia,
        "relativa_todos_tokens": frequencia / total_tokens,
        "relativa_tokens_conteudo": frequencia / total_tokens_conteudo,
    }
    for palavra, frequencia in frequencias_conteudo.most_common(12)
])
tabela_frequencias

## Concordâncias

Uma concordância recupera uma janela de $j$ tokens à esquerda e à direita de
cada ocorrência. Não é necessário transformar essa operação em uma medida única:
seu papel é devolver contexto ao agregado. As janelas devem respeitar as fronteiras
dos documentos e conservar o identificador da fonte.

In [ ]:
def concordancias(tabela, alvo, janela=4):
    resultados = []
    for _, documento in tabela.iterrows():
        tokens = documento["tokens"]
        for posicao, token in enumerate(tokens):
            if token == alvo:
                inicio = max(0, posicao - janela)
                fim = min(len(tokens), posicao + janela + 1)
                resultados.append({
                    "id_documento": documento["id_documento"],
                    "contexto": " ".join(tokens[inicio:fim]),
                })
    return pd.DataFrame(resultados)

concordancias(dados, "trabalho", janela=4).head(8)

## N-gramas e colocações

Um bigrama é o par adjacente $(t_i,t_{i+1})$ dentro de um mesmo documento.
Para comparar a frequência conjunta com as frequências marginais das posições
esquerda e direita, usaremos informação mútua pontual:

$$
PMI(a,b)=\log_2\left(\frac{P(a,b)}{P_L(a)P_R(b)}\right)
=\log_2\left(\frac{c(a,b)\,N_b}{c_L(a)c_R(b)}\right).
$$

| Símbolo | Significado |
|---|---|
| $c(a,b)$ | frequência do bigrama $(a,b)$ |
| $N_b$ | total de bigramas dentro dos documentos |
| $c_L(a)$ | ocorrências de $a$ na posição esquerda dos bigramas |
| $c_R(b)$ | ocorrências de $b$ na posição direita dos bigramas |

PMI alto indica associação acima do esperado pelas marginais; não indica
causalidade, importância histórica ou estabilidade. Como a medida favorece eventos
raros, exigiremos frequência mínima e retornaremos às concordâncias.

In [ ]:
bigramas = Counter()
marginal_esquerda = Counter()
marginal_direita = Counter()

for tokens_documento in dados["tokens"]:
    pares_documento = list(zip(tokens_documento, tokens_documento[1:]))
    bigramas.update(pares_documento)
    marginal_esquerda.update(a for a, _ in pares_documento)
    marginal_direita.update(b for _, b in pares_documento)

total_bigramas = sum(bigramas.values())
frequencia_minima = 3
linhas_pmi = []
for (a, b), frequencia in bigramas.items():
    if frequencia >= frequencia_minima:
        pmi = math.log2(
            frequencia * total_bigramas
            / (marginal_esquerda[a] * marginal_direita[b])
        )
        linhas_pmi.append((f"{a} {b}", frequencia, pmi))

tabela_colocacoes = pd.DataFrame(
    sorted(linhas_pmi, key=lambda linha: linha[2], reverse=True),
    columns=["bigrama", "frequencia", "pmi"],
)
tabela_colocacoes.head(10)

## Vocabulário e diversidade lexical

Se $V_d$ é o número de formas distintas e $N_d$ o número de tokens do
documento $d$, a razão forma–token é:

$$
TTR(d)=\frac{V_d}{N_d}.
$$

Como a TTR tende a cair quando o texto cresce, podemos comparar segmentos de
tamanho fixo $m$:

$$
TTR_m(d)=\frac{\left|\{t_1,\ldots,t_m\}\right|}{m},
\qquad N_d\geq m.
$$

Para que todos os 24 documentos participem, adotaremos como $m$ o tamanho do
menor texto da base. Usar os primeiros $m$ tokens controla o tamanho, mas ainda é
sensível à posição do trecho; projetos reais devem comparar segmentos ou amostras
com um protocolo explícito.

In [ ]:
tamanho_padrao = int(dados["tokens"].map(len).min())

def ttr_padronizada(tokens, tamanho):
    if len(tokens) < tamanho:
        return pd.NA
    segmento = tokens[:tamanho]
    return len(set(segmento)) / tamanho

diversidade = pd.DataFrame({
    "id_documento": dados["id_documento"],
    "tokens": dados["tokens"].map(len),
    "formas": dados["tokens"].map(lambda tokens: len(set(tokens))),
    "ttr": dados["tokens"].map(lambda tokens: len(set(tokens)) / len(tokens)),
    f"ttr_{tamanho_padrao}": dados["tokens"].map(
        lambda tokens: ttr_padronizada(tokens, tamanho_padrao)
    ),
})
print("Tamanho comum adotado:", tamanho_padrao, "tokens")
diversidade.head()

## Atividade
Documente regras, frequências, concordâncias, n-gramas, colocação e diversidade. Retorne a trechos. Escreva aqui.